In [42]:
import cv2
import numpy as np
import pickle
import mediapipe as mp
import time
import random

# Load model and label encoder
svm_model = pickle.load(open("best_model_svm.pkl", "rb"))
label_encoder = pickle.load(open("label_encoder.pkl", "rb"))

# MediaPipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
mp_drawing = mp.solutions.drawing_utils

# Drawing setup
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
current_color = colors[0]
drawing = False
last_pos = None
last_gesture = None
gesture_stable_count = 0

# Initialize video capture
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Failed to capture video")
    exit()

# Get first frame to initialize VideoWriter
ret, frame = cap.read()
if not ret:
    print("Failed to get initial frame")
    exit()

frame_height, frame_width = frame.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output.mp4", fourcc, 30, (frame_width, frame_height))

# Initialize canvas
canvas = np.ones((frame_height, frame_width, 3), dtype=np.uint8) * 255

def change_color():
    global current_color
    current_color = random.choice(colors)
    print("Color changed to:", current_color)

def clear_canvas():
    global canvas
    canvas = np.ones((frame_height, frame_width, 3), dtype=np.uint8) * 255
    print("Canvas cleared")

def save_canvas():
    filename = f"drawing_{int(time.time())}.png"
    cv2.imwrite(filename, canvas)
    print(f"Canvas saved as {filename}")

def preprocess_landmarks(hand_landmarks, frame_shape):
    """Process landmarks exactly like during training"""
    landmarks = np.array([(lm.x, lm.y, lm.z) for lm in hand_landmarks.landmark])
    
    # Normalize relative to wrist
    wrist_x, wrist_y, wrist_z = landmarks[0]
    landmarks[:, 0] -= wrist_x
    landmarks[:, 1] -= wrist_y
    
    # Normalize scale using middle finger
    mid_finger_x, mid_finger_y, _ = landmarks[12]
    scale_factor = np.sqrt(mid_finger_x**2 + mid_finger_y**2)
    if scale_factor > 0:
        landmarks[:, 0] /= scale_factor
        landmarks[:, 1] /= scale_factor
    
    return landmarks.flatten().reshape(1, -1)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to capture frame")
        break
        
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    # Process frame
    results = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    current_gesture = "none"  # Default
    
    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        try:
            # Preprocess landmarks exactly like during training
            features = preprocess_landmarks(hand_landmarks, frame.shape)
            
            # Get prediction
            numeric_prediction = svm_model.predict(features)[0]
            current_gesture = label_encoder.inverse_transform([numeric_prediction])[0]
            
            # Debug output
            print(f"Raw prediction: {numeric_prediction}, Gesture: {current_gesture}")
            
            # Gesture stabilization - require 3 consistent readings
            if current_gesture == last_gesture:
                gesture_stable_count += 1
            else:
                gesture_stable_count = 0
                last_gesture = current_gesture
            
            # Only execute actions after stable detection
            if gesture_stable_count >= 3:
                # Gesture actions
                if current_gesture == "like":
                    drawing = True
                elif current_gesture == "stop":
                    drawing = False
                    last_pos = None
                elif current_gesture == "fist":
                    change_color()
                elif current_gesture == "rock":
                    clear_canvas()
                    drawing = False
                elif current_gesture == "dislike":  # Fixed variable name here
                    save_canvas()
            
        except Exception as e:
            print(f"Prediction error: {str(e)}")
            current_gesture = "error"

        # Get index finger position for drawing
        index_tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
        finger_pos = (int(index_tip.x * w), int(index_tip.y * h))
        
        # Drawing logic
        if drawing and last_pos is not None:
            cv2.line(canvas, last_pos, finger_pos, current_color, 5)
        last_pos = finger_pos if drawing else None

    # Display info
    cv2.putText(frame, f"Gesture: {current_gesture}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.putText(frame, f"Drawing: {'ON' if drawing else 'OFF'}", (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.putText(frame, f"Color: {current_color}", (10, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 1, current_color, 2)

    # Combine frames
    combined = cv2.addWeighted(frame, 0.7, canvas, 0.3, 0)
    
    # Write the combined frame to output video
    out.write(combined)
    
    cv2.imshow("Touchless Whiteboard", combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release everything
cap.release()
out.release()
cv2.destroyAllWindows()
print("Resources released. Video saved as output.mp4")

I0000 00:00:1744307182.248776 1425050 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M1
W0000 00:00:1744307182.259435 1514614 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1744307182.273221 1514614 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3

Raw prediction: 4, Gesture: like
Raw prediction: 4, Gesture: like
Raw prediction: 4, Gesture: like
Raw prediction: 4, Gesture: like


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


Raw prediction: 4, Gesture: like
Raw prediction: 4, Gesture: like
Raw prediction: 4, Gesture: like
Raw prediction: 4, Gesture: like


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


...

Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature nam

Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Raw prediction: 11, Gesture: rock
Canvas cleared
Resources released. Video saved as output.mp4
